In [1]:
pip install -q langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [2]:
import os
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]="RAG_ADVANCED"
os.environ["LANGCHAIN_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""

In [3]:
from langsmith import utils
utils.tracing_is_enabled()

True

In [4]:
import bs4
import uuid
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [7]:
chain = (
    {"doc": lambda x: x.page_content} |
    ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}") |
    ChatOpenAI(temperature=0, model="gpt-4o-mini") |
    StrOutputParser()
)

summaries = chain.batch(docs, {"max_concurrency": 5})


In [8]:
from langchain.storage import InMemoryByteStore
from langchain.retrievers.multi_vector import MultiVectorRetriever


vectorstore = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())
store = InMemoryByteStore()
id_key = "doc_id"

/tmp/ipython-input-8-241954196.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())


In [9]:
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key
)

In [10]:
from langchain_core.documents import Document

doc_ids = [str(uuid.uuid4()) for _ in docs]

summary_docs = [
    Document(page_content=s, metadata={id_key:doc_ids[i]}) for i, s in enumerate(summaries)
]

In [11]:
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

In [12]:
query = "Memory in agents"
sub_docs = vectorstore.similarity_search(query, k=1)
sub_docs[0]

Document(metadata={'doc_id': '676bf8ec-6b5c-4ae9-8c12-88959b85e1ed'}, page_content='The document titled "LLM Powered Autonomous Agents" by Lilian Weng provides an in-depth exploration of autonomous agents that utilize large language models (LLMs) as their core controllers. It outlines the architecture and components necessary for building such agents, including planning, memory, and tool use.\n\n### Key Components:\n\n1. **Planning**:\n   - **Task Decomposition**: Agents break down complex tasks into smaller subgoals using techniques like Chain of Thought (CoT) and Tree of Thoughts (ToT) to enhance reasoning and problem-solving.\n   - **Self-Reflection**: Agents can evaluate their past actions to improve future performance, utilizing frameworks like ReAct and Reflexion to integrate reasoning with action.\n\n2. **Memory**:\n   - **Types of Memory**: The document categorizes memory into sensory, short-term, and long-term, drawing parallels to how LLMs manage information.\n   - **Maximum 

In [13]:
retrieved_docs = retriever.invoke(query, n_results = 1)
retrieved_docs[0].page_content[0:500]

"\n\n\n\n\n\nLLM Powered Autonomous Agents | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three:"